In [ ]:
# Task 3
# Workflow
# Use qc filtered dataset, compute hvg and select perturbations (all) present in 3 contitions with > 20 cells present each perturbation
# Select 50 perturbations regarding pathway information

In [2]:
# import packages
import scanpy as sc
import numpy as np
import pandas as pd

In [3]:
#Load the data
adata = sc.read_h5ad("../data/frangieh/adata_qc_done")

In [4]:
#Print the column names
print(adata.obs.columns)

Index(['library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA',
       'UMI_count', 'guide_id', 'perturbation', 'tissue_type', 'cancer',
       'disease', 'perturbation_type', 'celltype', 'organism',
       'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito',
       'percent_ribo', 'n_genes_by_counts', 'log1p_n_genes_by_counts',
       'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes',
       'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes',
       'pct_counts_in_top_500_genes', 'total_counts_mt',
       'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo',
       'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb',
       'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts',
       'doublet_scores', 'predicted_doublets', 'doublet_info'],
      dtype='object')


In [7]:
#Check the distribution of cells across the Control, IFNγ, and Co-culture conditions
print(adata.obs["perturbation_2"].value_counts())

perturbation_2
IFNγ          74186
Co-culture    67093
Control       49895
Name: count, dtype: int64


In [78]:
# Define the variables
RANDOM_STATE = 42
MIN_CELLS = 20
N_PERTURBATIONS = 50
N_TEST = 10
N_HVG = 2000
PERT_COL = "perturbation"
COND_COL = "perturbation_2"
CONDITIONS = ["Control", "IFNG", "Coculture"]

In [79]:
# Cell counts for perturbation x condition
counts = (
    adata.obs
    .groupby([PERT_COL, COND_COL], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Keep only the conditions we use
available_conditions = [
    c for c in CONDITIONS
    if c in counts.columns
]

print("Available conditions:", available_conditions)

# Keep perturbations with >= MIN_CELLS in every condition
eligible = counts[
    (counts[available_conditions] >= MIN_CELLS).all(axis=1)
].index.tolist()

print("Eligible perturbations:", len(eligible))

Available conditions: ['Control']
Eligible perturbations: 235


In [10]:
# Define possible labels used for unperturbed or non-targeting control cells
CONTROL_NAMES = {
    "control",
    "ctrl",
    "nt",
    "ntc",
    "non-targeting",
    "nontargeting",
    "non_targeting"
}

# Remove control/non-targeting labels from the list of eligible perturbations,
# since only actual gene perturbations should be considered for the 50-gene selection
eligible = [
    gene for gene in eligible
    if str(gene).lower() not in CONTROL_NAMES
]

# Display the number of eligible gene perturbations remaining after control removal
print("Eligible real perturbations:", len(eligible))

Eligible real perturbations: 234


In [13]:
import sys
!{sys.executable} -m pip install gseapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 14.8 MB/s  0:00:00


In [14]:
import gseapy as gp
print(gp.__version__)

1.3.1


In [15]:
#Download Hallmark pathway annotations
import gseapy as gp
hallmark = gp.get_library(
    name="MSigDB_Hallmark_2020",
    organism="Human"
)
print("Number of pathways:", len(hallmark))

Number of pathways: 50


In [16]:
# Convert all gene names in the Hallmark pathway dictionary to uppercase
# to ensure consistent gene-name matching regardless of capitalization
hallmark_upper = {
    pathway: {str(g).upper() for g in genes}
    for pathway, genes in hallmark.items()
}

In [17]:
import pandas as pd

# Create a dictionary mapping each eligible perturbation gene
# to its uppercase version for consistent matching with Hallmark gene names
eligible_upper = {
    gene: str(gene).upper()
    for gene in eligible
}

# Initialize a binary pathway-membership matrix:
# rows = eligible perturbation genes
# columns = Hallmark pathways
# 0 = gene is not part of the pathway
# 1 = gene is part of the pathway
pathway_matrix = pd.DataFrame(
    0,
    index=eligible,
    columns=hallmark_upper.keys(),
    dtype=int
)

# Check each eligible perturbation gene against every Hallmark pathway
for gene in eligible:

    # Retrieve the uppercase version of the perturbation gene
    gene_upper = eligible_upper[gene]

    for pathway, pathway_genes in hallmark_upper.items():

        # Mark the gene as 1 if it belongs to the corresponding Hallmark pathway
        if gene_upper in pathway_genes:
            pathway_matrix.loc[gene, pathway] = 1

In [18]:
# Display the dimensions of the pathway-membership matrix
# (number of eligible perturbation genes × number of Hallmark pathways)
print(pathway_matrix.shape)

# Display the first five perturbation genes and their binary pathway memberships
# to verify that the pathway annotation matrix was constructed correctly
pathway_matrix.head()

(234, 50)


,TNF-alpha Signaling via NF-kB,Hypoxia,Cholesterol Homeostasis,Mitotic Spindle,Wnt-beta Catenin Signaling,TGF-beta Signaling,IL-6/JAK/STAT3 Signaling,DNA Repair,G2-M Checkpoint,Apoptosis,...,heme Metabolism,Coagulation,IL-2/STAT5 Signaling,Bile Acid Metabolism,Pperoxisome,Allograft Rejection,Spermatogenesis,KRAS Signaling Up,KRAS Signaling Dn,Pancreas Beta Cells
A2M,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
ACSL3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ACTA2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AEBP1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AGA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
#Count how many Hallmark pathways annotate each perturbation:
gene_pathway_count = pathway_matrix.sum(axis=1)
gene_pathway_count.sort_values(
    ascending=False
).head(20)

CDKN1A    10
MYC       10
CD44       8
CCND2      7
TIMP1      7
CCND1      7
TGFB1      7
FOS        6
IFNGR2     5
CDK4       5
JAK2       5
KLF4       5
SAT1       5
NOLC1      5
CXCR4      5
IFNGR1     5
FKBP4      4
SGK1       4
MDH2       4
IDH2       4
dtype: int64

In [20]:
#how many eligible perturbations are represented in each pathway:
pathway_gene_count = pathway_matrix.sum(axis=0)
pathway_gene_count.sort_values(
    ascending=False
)

Myc Targets V1                       16
E2F Targets                          15
Allograft Rejection                  15
Interferon Gamma Response            14
p53 Pathway                          14
Estrogen Response Late               13
Apoptosis                            13
G2-M Checkpoint                      12
KRAS Signaling Up                    12
Complement                           12
IL-2/STAT5 Signaling                 11
TNF-alpha Signaling via NF-kB        10
Epithelial Mesenchymal Transition    10
mTORC1 Signaling                      9
Coagulation                           9
Glycolysis                            9
Estrogen Response Early               8
Xenobiotic Metabolism                 8
Oxidative Phosphorylation             8
Myogenesis                            7
Adipogenesis                          7
Androgen Response                     7
Hypoxia                               7
IL-6/JAK/STAT3 Signaling              7
Inflammatory Response                 7


In [21]:
#check genes with no Hallmark annotation
no_pathway = pathway_matrix.index[
    pathway_matrix.sum(axis=1) == 0
].tolist()

print("Eligible genes without Hallmark annotation:")
print(no_pathway)

print(
    "Annotated:",
    len(eligible) - len(no_pathway),
    "/",
    len(eligible)
)

Eligible genes without Hallmark annotation:
['AGA', 'ARMC6', 'ATP5MD', 'BOLA2B', 'BZW2', 'C6orf226', 'C19orf48', 'CCR10', 'CD58', 'CDC123', 'CDH19', 'CGAS', 'CHCHD2', 'CITED1', 'CMSS1', 'CSPG4', 'CST3', 'CTSA', 'DAG1', 'DDR1', 'DDX17', 'DLL3', 'EEA1', 'EEF1G', 'EIF3K', 'EVA1A', 'FBXO32', 'FMN1', 'FOXM1', 'FRZB', 'FTH1', 'GAS5', 'GPATCH4', 'GRN', 'GSEC', 'HASPIN', 'HLA-DRB5', 'HLA-H', 'IDI2-AS1', 'IRF3', 'ISYNA1', 'JMJD7', 'KDR', 'LEF1-AS1', 'LINC00518', 'LRRC75A-AS1', 'MC1R', 'MFGE8', 'MIA', 'MRPL47', 'NDUFA13', 'NEAT1', 'NGFR', 'NMRK1', 'NONO', 'NPC2', 'NSG1', 'NUP50-AS1', 'PET100', 'PFDN4', 'PIK3IP1', 'PSAP', 'PSMB8-AS1', 'PTMA', 'PUF60', 'RBP7', 'ROMO1', 'RPSA', 'RRS1', 'S100A6', 'S100B', 'SAE1', 'SCARB2', 'SEC11C', 'SERPINF1', 'SINHCAF', 'SLC5A3', 'SLC7A5P1', 'SLIRP', 'SMAD4', 'SNHG6', 'SOX4', 'SP100', 'SPESP1', 'SPINT1', 'SSR2', 'ST3GAL6-AS1', 'TAPBPL', 'TFAP2A', 'TMEM173', 'TOP1MT', 'TPP1', 'TRIM22', 'TSC22D3', 'TTLL1', 'TXNDC17', 'UCN2', 'VAT1', 'WBP2', 'XAGE1A']
Annotated: 134 

In [22]:
# Select 50 perturbations that maximize pathway coverage
# 134 from 234 perturbations are annotated to one pathway
import numpy as np
rng = np.random.default_rng(42)
annotated_genes = pathway_matrix.index[
    pathway_matrix.sum(axis=1) > 0
].tolist()
selected_50 = []
covered_pathways = set()
remaining = annotated_genes.copy()

In [23]:
# Iteratively select perturbation genes until 50 genes have been chosen
# or no eligible genes remain
while len(selected_50) < 50 and len(remaining) > 0:

    # Store how many previously uncovered pathways each remaining gene would add
    gains = {}

    for gene in remaining:

        # Identify all Hallmark pathways associated with the current gene
        gene_pathways = set(
            pathway_matrix.columns[
                pathway_matrix.loc[gene] == 1
            ]
        )

        # Determine which of these pathways have not yet been represented
        # by the already selected perturbations
        new_pathways = gene_pathways - covered_pathways

        # Record the number of new pathways that this gene would contribute
        gains[gene] = len(new_pathways)

    # Find the largest number of new pathways contributed by any remaining gene
    max_gain = max(gains.values())

    # Identify all genes that provide the maximum additional pathway coverage
    best_genes = [
        gene for gene, gain in gains.items()
        if gain == max_gain
    ]

    # Randomly choose one gene if several genes provide the same maximum gain
    # (using the predefined random generator for reproducibility)
    chosen = rng.choice(best_genes)

    # Add the selected gene to the final set of perturbations
    selected_50.append(chosen)

    # Retrieve all Hallmark pathways associated with the newly selected gene
    gene_pathways = set(
        pathway_matrix.columns[
            pathway_matrix.loc[chosen] == 1
        ]
    )

    # Update the set of pathways already represented by the selected genes
    covered_pathways.update(gene_pathways)

    # Remove the selected gene so that it cannot be selected again
    remaining.remove(chosen)

In [24]:
# If the pathway-based selection produced fewer than 50 perturbations,
# fill the remaining positions with other eligible perturbation genes
if len(selected_50) < 50:

    # Identify eligible genes that have not already been selected
    remaining_eligible = [
        gene for gene in eligible
        if gene not in selected_50
    ]

    # Randomly select enough additional genes to reach a total of 50
    # without selecting the same gene more than once
    extra = rng.choice(
        remaining_eligible,
        size=50 - len(selected_50),
        replace=False
    )

    # Add the randomly selected genes to the final perturbation set
    selected_50.extend(extra.tolist())

In [25]:
#Show selected perturbations and hallmark pathways represented
print("Selected perturbations:", len(selected_50))

print(
    "Hallmark pathways represented:",
    pathway_matrix.loc[selected_50]
    .max(axis=0)
    .sum()
)

Selected perturbations: 50
Hallmark pathways represented: 46


In [26]:
# Count how many of the 50 selected perturbation genes are associated
# with each Hallmark pathway
selected_pathway_counts = (
    pathway_matrix
    .loc[selected_50]             # Keep only the 50 selected perturbation genes
    .sum(axis=0)                  # Count pathway memberships across these genes
    .sort_values(ascending=False) # Sort pathways from most to least represented
)

# Display only Hallmark pathways represented by at least one
# of the 50 selected perturbation genes
selected_pathway_counts[
    selected_pathway_counts > 0
]

Myc Targets V1                       8
Estrogen Response Late               7
E2F Targets                          7
Allograft Rejection                  6
TNF-alpha Signaling via NF-kB        6
Androgen Response                    5
Oxidative Phosphorylation            5
G2-M Checkpoint                      5
Apoptosis                            5
p53 Pathway                          5
Complement                           4
Inflammatory Response                4
Coagulation                          4
Estrogen Response Early              4
Interferon Alpha Response            3
Myogenesis                           3
Interferon Gamma Response            3
heme Metabolism                      3
KRAS Signaling Up                    2
Pperoxisome                          2
Bile Acid Metabolism                 2
Glycolysis                           2
IL-2/STAT5 Signaling                 2
UV Response Up                       2
Adipogenesis                         2
Protein Secretion        

In [27]:
# Initialize the list of perturbation genes that will be held out for testing
# and a set to keep track of the Hallmark pathways already represented in the test set
test_genes = []
covered_test_pathways = set()

# Start with all 50 selected perturbation genes as possible test candidates
remaining = selected_50.copy()

# Iteratively select 10 held-out test perturbations
# while maximizing pathway diversity in the test set
while len(test_genes) < 10:

    # Store how many previously uncovered pathways each candidate gene would add
    gains = {}

    for gene in remaining:

        # Identify all Hallmark pathways associated with the current gene
        gene_pathways = set(
            pathway_matrix.columns[
                pathway_matrix.loc[gene] == 1
            ]
        )

        # Calculate how many new pathways this gene would contribute
        # compared with pathways already represented in the test set
        gains[gene] = len(
            gene_pathways - covered_test_pathways
        )

    # Find the maximum number of new pathways contributed by any remaining gene
    max_gain = max(gains.values())

    # Identify all genes that provide the maximum additional pathway coverage
    candidates = [
        gene for gene, gain in gains.items()
        if gain == max_gain
    ]

    # If several genes provide the same pathway gain, randomly select one
    # using the predefined random generator for reproducibility
    chosen = rng.choice(candidates)

    # Add the selected perturbation gene to the held-out test set
    test_genes.append(chosen)

    # Update the set of Hallmark pathways represented by the test genes
    covered_test_pathways.update(
        pathway_matrix.columns[
            pathway_matrix.loc[chosen] == 1
        ]
    )

    # Remove the selected gene from the candidate pool
    # so that it cannot be selected again
    remaining.remove(chosen)

In [28]:
# Create the training set by selecting all genes from the 50 selected perturbations
# that were not assigned to the held-out test set
train_genes = [
    gene for gene in selected_50
    if gene not in test_genes
]

# Check that the final split contains 40 training perturbations
# and 10 completely held-out test perturbations
print("Train:", len(train_genes))
print("Test:", len(test_genes))

# Display the perturbation genes that will be used for model training
print("\nTrain genes:")
print(train_genes)

# Display the perturbation genes that will remain completely unseen during
# model training and hyperparameter optimization and will only be used for final testing
print("\nHeld-out test genes:")
print(test_genes)

Train: 40
Test: 10

Train genes:
[np.str_('LAMP2'), np.str_('SGK1'), np.str_('B2M'), np.str_('CDK6'), np.str_('BOLA2'), np.str_('KCNN4'), np.str_('TYR'), np.str_('PFN1'), np.str_('HLA-F'), np.str_('GSN'), np.str_('JPT1'), np.str_('CD47'), np.str_('ST6GALNAC2'), np.str_('HNRNPC'), np.str_('JAK1'), np.str_('FOS'), np.str_('CDKN2A'), np.str_('TMED10'), np.str_('SET'), np.str_('PPA1'), np.str_('GPNMB'), np.str_('SLC26A2'), np.str_('LRPAP1'), np.str_('HLA-E'), np.str_('HNRNPA1'), np.str_('CTPS1'), np.str_('ATP1B1'), np.str_('NCL'), np.str_('TIMM50'), np.str_('APOC2'), np.str_('STOM'), np.str_('SLC25A13'), np.str_('CYP27A1'), np.str_('POLD2'), np.str_('CD274'), np.str_('RB1'), np.str_('FBL'), np.str_('AHNAK'), np.str_('UQCRH'), np.str_('HLA-C')]

Held-out test genes:
[np.str_('MYC'), np.str_('CDKN1A'), np.str_('TIMP1'), np.str_('IDH2'), np.str_('MDH2'), np.str_('CCND1'), np.str_('SDCBP'), np.str_('IFNGR2'), np.str_('NOLC1'), np.str_('LGALS3')]


In [29]:
# Determine which Hallmark pathways are represented by at least one
# perturbation gene in the training set
train_coverage = (
    pathway_matrix
    .loc[train_genes]  # Keep only the 40 training perturbations
    .max(axis=0)       # 1 if at least one training gene belongs to the pathway
)

# Determine which Hallmark pathways are represented by at least one
# perturbation gene in the held-out test set
test_coverage = (
    pathway_matrix
    .loc[test_genes]   # Keep only the 10 held-out perturbations
    .max(axis=0)       # 1 if at least one test gene belongs to the pathway
)

# Count the total number of Hallmark pathways represented
# among the training perturbations
print(
    "Pathways represented in train:",
    int(train_coverage.sum())
)

# Count the total number of Hallmark pathways represented
# among the held-out test perturbations
print(
    "Pathways represented in test:",
    int(test_coverage.sum())
)

Pathways represented in train: 35
Pathways represented in test: 37


In [30]:
# Identify the Hallmark pathways represented by at least one of the 10 held-out test perturbation genes
test_pathways = test_coverage[
    test_coverage == 1
].index.tolist()

# Display the list of Hallmark pathways represented in the test set
print(test_pathways)

['TNF-alpha Signaling via NF-kB', 'Hypoxia', 'Cholesterol Homeostasis', 'Wnt-beta Catenin Signaling', 'TGF-beta Signaling', 'IL-6/JAK/STAT3 Signaling', 'DNA Repair', 'G2-M Checkpoint', 'Apoptosis', 'Notch Signaling', 'Adipogenesis', 'Estrogen Response Early', 'Estrogen Response Late', 'Androgen Response', 'Myogenesis', 'Interferon Gamma Response', 'Complement', 'Unfolded Protein Response', 'PI3K/AKT/mTOR  Signaling', 'mTORC1 Signaling', 'E2F Targets', 'Myc Targets V1', 'Myc Targets V2', 'Epithelial Mesenchymal Transition', 'Inflammatory Response', 'Fatty Acid Metabolism', 'Oxidative Phosphorylation', 'Glycolysis', 'p53 Pathway', 'UV Response Dn', 'Angiogenesis', 'heme Metabolism', 'Coagulation', 'IL-2/STAT5 Signaling', 'Bile Acid Metabolism', 'Pperoxisome', 'Allograft Rejection']


In [32]:
# Compute highly variable genes (HVGs) to define a reduced set of
# informative genes used as the transcriptomic response variables

import scanpy as sc
import numpy as np
from scipy import sparse

# Create a copy of the original AnnData object so that normalization
# and log transformation for HVG selection do not modify the original data
adata_hvg = adata.copy()

# Normalize gene-expression counts so that each cell has the same
# total expression count (10,000 counts per cell)
sc.pp.normalize_total(
    adata_hvg,
    target_sum=1e4
)

# Apply a log(1+x) transformation to the normalized expression values
# before identifying highly variable genes
sc.pp.log1p(adata_hvg)

# Identify the 2,000 genes showing the highest variability across cells
sc.pp.highly_variable_genes(
    adata_hvg,
    n_top_genes=2000
)

# Extract the names of the 2,000 selected highly variable genes
# that will be used to represent the transcriptomic response
response_hvgs = adata_hvg.var_names[
    adata_hvg.var["highly_variable"]
].tolist()

# Verify that 2,000 HVGs were selected and display the first 20 genes
print("Number of HVGs:", len(response_hvgs))
print(response_hvgs[:20])

Number of HVGs: 2000
['AADAC', 'AATBC', 'ABCB11', 'ABCB6', 'ABHD12B', 'ABL2', 'AC002310.2', 'AC002347.1', 'AC002384.1', 'AC003092.1', 'AC003956.1', 'AC004112.1', 'AC004231.1', 'AC004232.2', 'AC004448.2', 'AC004471.2', 'AC004528.1', 'AC004584.1', 'AC004771.4', 'AC004771.5']


In [33]:
# Use the 2,000 selected highly variable genes as the response genes
# whose mean log2 fold-change values will be predicted by the models
response_genes = response_hvgs

In [35]:
import numpy as np
from scipy import sparse

# Small constant added when calculating log2 fold changes later
# to avoid division by zero
PSEUDOCOUNT = 1e-3


# Define a function to calculate the mean expression of the selected
# response genes across all cells in an AnnData subset
def mean_expression(adata_subset, genes):

    # Extract the expression matrix for the selected response genes
    X = adata_subset[:, genes].X

    # Calculate the mean expression of each gene across all cells
    if sparse.issparse(X):
        mean = np.asarray(
            X.mean(axis=0)
        ).ravel()
    else:
        mean = np.asarray(
            X.mean(axis=0)
        ).ravel()

    # Return one mean expression value for each response gene
    return mean

In [36]:
# Display the 20 most frequent perturbation targets
# and the number of cells associated with each perturbation
print(
    adata.obs[PERT_COL]
    .value_counts()
    .head(20)
)

perturbation
control     49841
ACTA2        1215
B2M          1145
IFNGR2       1094
A2M          1092
AEBP1        1086
CTSD         1076
CD59         1046
CD44         1046
FGFR1        1035
CD274        1034
CSPG4        1018
CDH19        1017
JAK2         1016
CD58          999
C19orf48      966
APOC2         964
ATP1B1        953
AGA           947
APOE          943
Name: count, dtype: int64


In [40]:
# Define the label used to identify unperturbed/control cells
# in the perturbation column
UNPERTURBED_LABEL = "control"

# Define the three experimental conditions included in the analysis
CONDITIONS = ["Control", "IFNγ", "Co-culture"]

# Define the AnnData observation column containing the
# CRISPR perturbation target for each cell
PERT_COL = "perturbation"

# Define the AnnData observation column containing the
# experimental condition for each cell
COND_COL = "perturbation_2"

In [41]:
# Create a Boolean mask to identify unperturbed/control cells
# within the current experimental condition
baseline_mask = (
    
    # Select only cells belonging to the current condition
    # (Control, IFNγ, or Co-culture)
    (adata.obs[COND_COL] == condition)
    
    &
    
    # Select cells whose perturbation label corresponds to
    # the unperturbed/control population
    (
        adata.obs[PERT_COL]
        .astype(str)
        .str.lower()
        == UNPERTURBED_LABEL.lower()
    )
)

In [42]:
def mean_expression(adata_subset, genes):

    # Extract the expression matrix for the selected response genes
    X = adata_subset[:, genes].X

    # Calculate the mean expression of each gene across all cells
    if sparse.issparse(X):
        mean = np.asarray(X.mean(axis=0)).ravel()
    else:
        mean = np.asarray(X.mean(axis=0)).ravel()

    # Return one mean expression value for each response gene
    return mean

In [44]:
#Print number of perturbation profiles
print("Number of profiles:", len(profiles))

Number of profiles: 150


In [45]:
# Construct the target matrix Y containing the transcriptomic
# response (mean log2 fold change) for each perturbation-condition profile

import pandas as pd

# Create a metadata table describing each response profile.
# Each row corresponds to one perturbation under one experimental condition
# and stores the perturbation target, condition, and number of cells used.
profile_metadata = pd.DataFrame([
    {
        "perturbation": p["perturbation"],
        "condition": p["condition"],
        "n_cells": p["n_cells"]
    }
    for p in profiles
])

# Stack the log2FC response vectors into a single target matrix.
# Rows correspond to perturbation-condition profiles,
# while columns correspond to the 2,000 selected HVGs.
Y = np.vstack([
    p["log2fc"]
    for p in profiles
])

# Check the dimensions of the metadata table and target matrix.
# With 50 perturbations × 3 conditions, 150 profiles are expected.
# Y should therefore have 150 rows and 2,000 response genes.
print("Metadata:", profile_metadata.shape)
print("Target Y:", Y.shape)

Metadata: (150, 3)
Target Y: (150, 2000)


In [47]:
from scipy import sparse
import numpy as np

# Access the gene-expression matrix from the AnnData object
X = adata.X

# Extract the stored expression values depending on whether
# the expression matrix is sparse or dense
if sparse.issparse(X):
    values = X.data
else:
    values = np.asarray(X).ravel()

# Check the expression matrix for missing (NaN) or infinite values
# and inspect its numerical range
print("NaN:", np.isnan(values).sum())
print("Inf:", np.isinf(values).sum())
print("Min:", np.nanmin(values))
print("Max:", np.nanmax(values))

# Display the first 20 stored expression values to inspect
# the scale and format of the expression data
print("First values:")
print(values[:20])

NaN: 0
Inf: 0
Min: 1.0
Max: 4437.0
First values:
[2. 1. 1. 1. 1. 2. 1. 1. 2. 2. 1. 1. 2. 5. 1. 1. 1. 1. 2. 1.]


In [48]:
from scipy import sparse
import numpy as np

# Define a function to calculate the mean expression of the selected
# response genes across all cells in a given AnnData subset
def mean_expression(adata_subset, genes):

    # Extract the expression matrix for the selected genes
    X = adata_subset[:, genes].X

    # Calculate the mean expression of each gene across all cells.
    # Handle sparse and dense expression matrices appropriately.
    if sparse.issparse(X):
        mean = np.asarray(
            X.mean(axis=0)
        ).ravel()
    else:
        mean = np.asarray(
            X.mean(axis=0)
        ).ravel()

    # Return a one-dimensional vector containing one mean
    # expression value for each selected response gene
    return mean

In [49]:
# Initialize a list to store the condition-specific
# transcriptional response profiles for the 50 selected perturbations
profiles = []

# Loop through the three experimental conditions
for condition in CONDITIONS:

    # Identify unperturbed/control cells within the current condition
    baseline_mask = (
        (adata.obs[COND_COL] == condition)
        &
        (
            adata.obs[PERT_COL]
            .astype(str)
            .str.lower()
            == UNPERTURBED_LABEL.lower()
        )
    )

    # Extract the condition-matched unperturbed cells
    baseline_adata = adata[baseline_mask]

    # Display the number of baseline cells available
    # for the current experimental condition
    print(
        condition,
        "baseline cells:",
        baseline_adata.n_obs
    )

    # Calculate the mean expression of the 2,000 response genes
    # in the unperturbed cells of the current condition
    baseline_mean = mean_expression(
        baseline_adata,
        response_genes
    )

    # Loop through each of the 50 selected perturbation targets
    for pert in selected_50:

        # Identify cells carrying the current perturbation
        # within the current experimental condition
        pert_mask = (
            (adata.obs[COND_COL] == condition)
            &
            (adata.obs[PERT_COL] == pert)
        )

        # Extract the corresponding perturbed cells
        pert_adata = adata[pert_mask]

        # Skip perturbation-condition combinations with fewer
        # than the minimum required number of cells
        if pert_adata.n_obs < MIN_CELLS:
            continue

        # Calculate the mean expression of the 2,000 response genes
        # for the current perturbation-condition combination
        pert_mean = mean_expression(
            pert_adata,
            response_genes
        )

        # Calculate the mean log2 fold change for each response gene
        # relative to the condition-matched unperturbed baseline.
        # The pseudocount prevents division by zero.
        log2fc = np.log2(
            (pert_mean + PSEUDOCOUNT)
            /
            (baseline_mean + PSEUDOCOUNT)
        )

        # Store the perturbation name, experimental condition,
        # number of cells, and 2,000-gene log2FC response vector
        profiles.append({
            "perturbation": pert,
            "condition": condition,
            "n_cells": pert_adata.n_obs,
            "log2fc": log2fc
        })

Control baseline cells: 12941
IFNγ baseline cells: 20192
Co-culture baseline cells: 16708


In [50]:
# Create a metadata table describing each perturbation-condition profile.
# Each row corresponds to one of the generated transcriptional response profiles.
profile_metadata = pd.DataFrame([
    {
        # CRISPR perturbation target
        "perturbation": p["perturbation"],

        # Experimental condition: Control, IFNγ, or Co-culture
        "condition": p["condition"],

        # Number of cells used to calculate the mean expression profile
        "n_cells": p["n_cells"]
    }
    for p in profiles
])

# Construct the target matrix Y by stacking the log2FC vectors
# from all perturbation-condition profiles.
# Rows = perturbation-condition profiles
# Columns = the 2,000 selected highly variable response genes
Y = np.vstack([
    p["log2fc"]
    for p in profiles
])

In [52]:
# Initialize a list that will store the input feature vector
# for each perturbation-condition profile
X_rows = []

# Loop through each row of the profile metadata table
for _, row in profile_metadata.iterrows():

    # Get the perturbation target gene for the current profile
    gene = row["perturbation"]

    # Get the experimental condition for the current profile
    condition = row["condition"]

    # Retrieve the Hallmark pathway membership features
    # of the current perturbation gene
    features = pathway_matrix.loc[gene].copy()

    # Add the experimental condition as one-hot encoded features.
    # Exactly one of these three variables will be equal to 1.
    features["condition_Control"] = int(
        condition == "Control"
    )

    features["condition_IFNγ"] = int(
        condition == "IFNγ"
    )

    features["condition_Co-culture"] = int(
        condition == "Co-culture"
    )

    # Add the complete feature vector for this
    # perturbation-condition profile to the input list
    X_rows.append(features)


# Convert the list of feature vectors into the final input matrix X.
# Rows = perturbation-condition profiles
# Columns = Hallmark pathway features + experimental condition features
X = pd.DataFrame(
    X_rows
).reset_index(drop=True)

# Check that the input and target matrices contain
# the same number of perturbation-condition profiles
print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (150, 53)
Y shape: (150, 2000)


In [53]:
# Create Boolean masks that identify profiles belonging to the
# 40 training perturbations and the 10 held-out test perturbations
train_mask = profile_metadata["perturbation"].isin(train_genes)
test_mask = profile_metadata["perturbation"].isin(test_genes)


# Split the input feature matrix X according to the perturbation-level split.
# X_train contains pathway + condition features for the training perturbations,
# while X_test contains the same features for the unseen test perturbations.
X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[test_mask].reset_index(drop=True)


# Split the target matrix Y using the same masks.
# Each row contains the 2,000-gene log2FC response for the corresponding
# perturbation-condition profile.
Y_train = Y[train_mask.values]
Y_test = Y[test_mask.values]


# Keep the corresponding metadata for the training profiles.
# This is particularly useful later for GroupKFold, where perturbation
# target will be used as the grouping variable.
meta_train = profile_metadata.loc[
    train_mask
].reset_index(drop=True)


# Keep the metadata for the held-out test profiles for later
# evaluation by perturbation and experimental condition.
meta_test = profile_metadata.loc[
    test_mask
].reset_index(drop=True)


# Display the dimensions of the training input and target matrices.
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

# Display the dimensions of the held-out test input and target matrices.
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

X_train: (120, 53)
Y_train: (120, 2000)
X_test: (30, 53)
Y_test: (30, 2000)


In [54]:
# Check that no perturbation gene appears in both the training
# and held-out test sets, ensuring a strict perturbation-level split
print(
    "Overlap:",
    set(meta_train["perturbation"])
    &
    set(meta_test["perturbation"])
)

Overlap: set()


In [55]:
# Extract the Hallmark pathway-membership information
# for the 50 selected perturbation genes
selected_pathway_matrix = pathway_matrix.loc[selected_50]

# Identify selected perturbation genes that are not annotated
# to any of the Hallmark pathways.
# A row sum of 0 means that the gene has no Hallmark pathway membership.
zero_pathway_genes = selected_pathway_matrix.index[
    selected_pathway_matrix.sum(axis=1) == 0
].tolist()

# Display the selected perturbations for which no
# Hallmark pathway annotation was found
print(
    "Selected genes without Hallmark annotation:",
    zero_pathway_genes
)

Selected genes without Hallmark annotation: []


In [58]:
from sklearn.decomposition import PCA

# Define the number of principal components used to represent
# the 2,000-dimensional transcriptomic response
N_PCS = 50

# Initialize PCA to reduce the dimensionality of the target matrix Y
# from 2,000 gene-level log2FC values to 50 principal components
pca_y = PCA(
    n_components=N_PCS,
    random_state=42
)

# Fit PCA only on the training response profiles and transform
# Y_train into the 50-dimensional PCA representation.
# The test data are not used to learn the PCA components,
# preventing information leakage.
Y_train_pca = pca_y.fit_transform(Y_train)

# Project the held-out test response profiles onto the PCA space
# learned exclusively from the training data
Y_test_pca = pca_y.transform(Y_test)

# Display the dimensions before and after PCA transformation
print("Original Y_train:", Y_train.shape)
print("PCA Y_train:", Y_train_pca.shape)

# Calculate the total proportion of variance in the original
# 2,000-gene response profiles retained by the 50 PCs
print(
    "Variance explained:",
    pca_y.explained_variance_ratio_.sum()
)

Original Y_train: (120, 2000)
PCA Y_train: (120, 50)
Variance explained: 0.65633315


In [59]:
from sklearn.ensemble import RandomForestRegressor

# Initialize a Random Forest regression model to predict the
# 50 PCA components of the transcriptomic response
rf = RandomForestRegressor(

    # Number of decision trees included in the forest
    n_estimators=300,

    # Limit the maximum depth of each tree to reduce overfitting
    max_depth=6,

    # Require at least two training observations in each terminal leaf
    min_samples_leaf=2,

    # Set a fixed random seed to ensure reproducible results
    random_state=42,

    # Use all available CPU cores to speed up model training
    n_jobs=-1
)

# Train the Random Forest using:
# X_train = Hallmark pathway membership + experimental condition features
# Y_train_pca = transcriptomic response represented by 50 PCA components
rf.fit(
    X_train,
    Y_train_pca
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_

In [60]:
# Use the trained Random Forest to predict the transcriptomic
# responses of the 10 completely held-out perturbations.
# Predictions are made in the 50-dimensional PCA response space.
Y_pred_pca = rf.predict(X_test)

# Display the dimensions of the predicted response matrix
print(Y_pred_pca.shape)

(30, 50)


In [61]:
# Transform the Random Forest predictions from the 50-dimensional
# PCA space back into the original 2,000-gene log2FC response space
Y_pred = pca_y.inverse_transform(
    Y_pred_pca
)
# Display the dimensions of the reconstructed predicted response matrix
print(Y_pred.shape)

(30, 2000)


In [62]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# Calculate the overall Mean Absolute Error (MAE) between the
# observed and predicted log2FC values across all held-out profiles
# and all 2,000 response genes
mae = mean_absolute_error(
    Y_test.ravel(),
    Y_pred.ravel()
)

# Calculate the overall Root Mean Squared Error (RMSE).
# RMSE gives greater weight to large prediction errors than MAE.
rmse = np.sqrt(
    mean_squared_error(
        Y_test.ravel(),
        Y_pred.ravel()
    )
)

# Display the overall prediction errors on the held-out test set
print("Test MAE:", mae)
print("Test RMSE:", rmse)

Test MAE: 1.9138914099488689
Test RMSE: 3.470880359013139


In [63]:
from scipy.stats import pearsonr

# Initialize a list to store the Pearson correlation coefficient
# for each held-out perturbation-condition response profile
correlations = []

# Loop through all test profiles
# (10 held-out perturbations × 3 conditions = 30 profiles)
for i in range(Y_test.shape[0]):

    # Calculate the Pearson correlation between the observed and
    # predicted log2FC values across the 2,000 response genes
    # for the current perturbation-condition profile
    r, _ = pearsonr(
        Y_test[i],
        Y_pred[i]
    )

    # Store the correlation coefficient for this profile
    correlations.append(r)

# Calculate and display the average Pearson correlation
# across all held-out perturbation-condition profiles
print(
    "Mean Pearson correlation:",
    np.mean(correlations)
)

# Calculate and display the median Pearson correlation.
# The median is less sensitive to unusually high or low correlations.
print(
    "Median Pearson correlation:",
    np.median(correlations)
)

Mean Pearson correlation: 0.6927286270359309
Median Pearson correlation: 0.6877811992438244


In [66]:
# Create a copy of the held-out test metadata containing the
# perturbation target, experimental condition, and cell count
results = meta_test.copy()

# Add the Pearson correlation coefficient calculated for each
# perturbation-condition profile.
# This measures the similarity between the observed and predicted
# 2,000-gene log2FC response patterns for each test profile.
results["pearson_r"] = correlations

# Display the first 10 test profiles together with their
# corresponding Pearson correlation values
results.head(10)

,perturbation,condition,n_cells,pearson_r
0,MYC,Control,37,0.762933
1,CDKN1A,Control,254,0.673076
2,TIMP1,Control,137,0.672943
3,IDH2,Control,229,0.629739
4,MDH2,Control,153,0.687333
5,CCND1,Control,163,0.671190
6,SDCBP,Control,162,0.701179
7,IFNGR2,Control,193,0.694632
8,NOLC1,Control,70,0.731704
9,LGALS3,Control,161,0.676525


In [67]:
# Summarize the Pearson correlation values separately
# for each experimental condition
print(
    results
    .groupby("condition")["pearson_r"]

    # Calculate the mean, median, and standard deviation
    # of the profile-level Pearson correlations within each condition
    .agg(["mean", "median", "std"])
)

                mean    median       std
condition                               
Co-culture  0.711310  0.702697  0.055213
Control     0.690126  0.681929  0.036448
IFNγ        0.676751  0.665144  0.038486


In [68]:
# Summarize prediction performance separately for each
# of the 10 held-out perturbation genes
print(
    results
    # Group the three condition-specific profiles
    # (Control, IFNγ, and Co-culture) by perturbation target
    .groupby("perturbation")["pearson_r"]
    # Calculate the mean Pearson correlation across the three conditions
    # and its standard deviation for each held-out perturbation
    .agg(["mean", "std"])
    # Rank the held-out perturbations from best to worst predicted
    # according to their mean Pearson correlation
    .sort_values(
        "mean",
        ascending=False
    )
)

                  mean       std
perturbation                    
MYC           0.775676  0.032765
NOLC1         0.741258  0.023365
LGALS3        0.707097  0.038724
SDCBP         0.692206  0.031922
MDH2          0.691900  0.007148
CCND1         0.679489  0.022782
TIMP1         0.673186  0.005043
CDKN1A        0.661757  0.016237
IDH2          0.659921  0.032062
IFNGR2        0.644797  0.043281


In [69]:
from sklearn.model_selection import (
    GroupKFold,
    RandomizedSearchCV
)

# Use the perturbation target as the grouping variable so that
# all three condition-specific profiles of the same perturbation
# remain together during cross-validation
groups = meta_train[
    "perturbation"
].values

# Define 5-fold grouped cross-validation.
# This prevents the same perturbation gene from appearing in both
# the training and validation portions of a fold.
cv = GroupKFold(
    n_splits=5
)

# Define the hyperparameter search space for the Random Forest regressor
param_dist = {

    # Number of trees in the forest
    "n_estimators": [100, 200, 300, 400],

    # Maximum depth allowed for each decision tree.
    # None means trees can grow until another stopping criterion is reached.
    "max_depth": [3, 6, 9, 12, None],

    # Minimum number of training observations required
    # to split an internal tree node
    "min_samples_split": [2, 4, 6, 8],

    # Minimum number of training observations allowed
    # in a terminal leaf node
    "min_samples_leaf": [1, 2, 3, 4],

    # Number or proportion of input features considered
    # when searching for the best split at each tree node
    "max_features": ["sqrt", 0.5, 1.0]
}

In [70]:
# Initialize the Random Forest regressor.
# The hyperparameters will be selected by RandomizedSearchCV.
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Set up randomized hyperparameter optimization
search = RandomizedSearchCV(

    # Model whose hyperparameters will be optimized
    estimator=rf,

    # Hyperparameter combinations to sample from
    param_distributions=param_dist,

    # Randomly evaluate 50 different hyperparameter combinations
    n_iter=50,

    # Use negative mean squared error as the cross-validation score.
    # Scikit-learn maximizes scores, so MSE is expressed as a negative value.
    scoring="neg_mean_squared_error",

    # Use the previously defined 5-fold GroupKFold cross-validation
    cv=cv,

    # Fix the random seed so the randomized search is reproducible
    random_state=42,

    # Use all available CPU cores
    n_jobs=-1,

    # Display progress during hyperparameter optimization
    verbose=1
)

# Perform hyperparameter optimization using only the training data.
# The grouping variable ensures that all three condition-specific
# profiles of the same perturbation remain in the same CV fold.
search.fit(
    X_train,
    Y_train_pca,
    groups=groups
)

# Display the hyperparameter combination that achieved the
# best cross-validation performance
print(
    "Best parameters:",
    search.best_params_
)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'n_estimators': 400, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.5, 'max_depth': 3}


In [72]:
# Retrieve the Random Forest model with the best hyperparameter
# combination identified by RandomizedSearchCV.
# This model has already been refitted on the complete training set.
best_rf = search.best_estimator_

# Use the optimized Random Forest to predict the transcriptomic
# responses of the 10 completely held-out perturbations.
# Predictions are initially produced in the 50-dimensional PCA space.
Y_pred_pca = best_rf.predict(
    X_test
)

# Transform the predicted PCA scores back into the original
# 2,000-gene log2FC response space so that predictions can be
# directly compared with the observed Y_test profiles.
Y_pred = pca_y.inverse_transform(
    Y_pred_pca
)

In [73]:
# Use the optimized Random Forest to predict the 50 PCA components
# of the transcriptomic response for the held-out test perturbations
Y_pred_pca = best_rf.predict(X_test)

# Transform the predicted PCA components back into the original
# 2,000-gene log2FC space for comparison with the observed responses
Y_pred = pca_y.inverse_transform(Y_pred_pca)

In [74]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

# Calculate the overall MAE across all held-out test profiles
# and all 2,000 response genes
mae = mean_absolute_error(
    Y_test.ravel(),
    Y_pred.ravel()
)

# Calculate the overall RMSE across all held-out test profiles
# and all 2,000 response genes
rmse = np.sqrt(
    mean_squared_error(
        Y_test.ravel(),
        Y_pred.ravel()
    )
)

print("Overall Test MAE:", mae)
print("Overall Test RMSE:", rmse)

# Initialize a list to store the Pearson correlation
# for each perturbation-condition profile
pearson_values = []

# Calculate Pearson correlation between the observed and predicted
# 2,000-gene log2FC profiles for each test profile
for i in range(Y_test.shape[0]):

    true_profile = Y_test[i]
    pred_profile = Y_pred[i]

    # Pearson correlation is undefined for constant profiles
    if np.std(true_profile) == 0 or np.std(pred_profile) == 0:
        r = np.nan
    else:
        r, _ = pearsonr(
            true_profile,
            pred_profile
        )

    pearson_values.append(r)

# Calculate the mean and median Pearson correlation
# across all held-out perturbation-condition profiles
print(
    "Mean Pearson r:",
    np.nanmean(pearson_values)
)

print(
    "Median Pearson r:",
    np.nanmedian(pearson_values)
)

# Create a results table containing the metadata
# for each held-out test profile
results = meta_test.copy()

# Calculate MAE separately for each perturbation-condition profile
results["MAE"] = [
    mean_absolute_error(
        Y_test[i],
        Y_pred[i]
    )
    for i in range(Y_test.shape[0])
]

# Calculate RMSE separately for each perturbation-condition profile
results["RMSE"] = [
    np.sqrt(
        mean_squared_error(
            Y_test[i],
            Y_pred[i]
        )
    )
    for i in range(Y_test.shape[0])
]

# Add the profile-specific Pearson correlations
results["Pearson_r"] = pearson_values

# Display the prediction performance for each
# held-out perturbation-condition profile
print("\nResults per test profile:")
print(results)

Overall Test MAE: 1.9392062512982786
Overall Test RMSE: 3.45014849867564
Mean Pearson r: 0.6984454500398699
Median Pearson r: 0.6955798189546458

Results per test profile:
   perturbation   condition  n_cells       MAE      RMSE  Pearson_r
0           MYC     Control       37  1.844552  3.329015   0.767216
1        CDKN1A     Control      254  2.118156  3.609752   0.683331
2         TIMP1     Control      137  2.130569  3.690949   0.681582
3          IDH2     Control      229  2.225858  3.877660   0.636269
4          MDH2     Control      153  2.059978  3.639523   0.688508
5         CCND1     Control      163  2.139743  3.743601   0.677418
6         SDCBP     Control      162  2.001410  3.535889   0.703001
7        IFNGR2     Control      193  2.031279  3.518510   0.698778
8         NOLC1     Control       70  1.903122  3.472828   0.733434
9        LGALS3     Control      161  2.117437  3.730648   0.679271
10          MYC        IFNγ       57  1.847135  3.300174   0.743772
11       CDK

In [75]:
# 4. Performance by condition
condition_results = (
    results
    .groupby("condition")
    [["MAE", "RMSE", "Pearson_r"]]
    .agg(["mean", "std", "median"])
)

print("\nPerformance by condition:")
print(condition_results)


Performance by condition:
                 MAE                          RMSE                      \
                mean       std    median      mean       std    median   
condition                                                                
Co-culture  1.752229  0.176254  1.764148  3.245677  0.219407  3.287699   
Control     2.057211  0.115824  2.088708  3.614837  0.157363  3.624638   
IFNγ        2.008179  0.113563  2.026922  3.468322  0.125783  3.461102   

           Pearson_r                      
                mean       std    median  
condition                                 
Co-culture  0.717984  0.052097  0.712451  
Control     0.694881  0.035174  0.685919  
IFNγ        0.682471  0.035124  0.682600  


In [76]:
# Calculate uncertainty of the test performance using
# perturbation-level bootstrap resampling

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

# Initialize a reproducible random number generator
rng = np.random.default_rng(42)

# Extract the names of the 10 unique held-out perturbation genes
test_gene_names = meta_test["perturbation"].unique()

# Define the number of bootstrap repetitions
N_BOOTSTRAPS = 1000

# Initialize lists to store the performance metric
# obtained from each bootstrap sample
bootstrap_mae = []
bootstrap_rmse = []
bootstrap_pearson = []

# Repeat the bootstrap procedure 1,000 times
for b in range(N_BOOTSTRAPS):

    # Randomly sample the 10 held-out perturbation genes with replacement.
    # Sampling at the perturbation level keeps all three condition-specific
    # profiles of a perturbation together.
    sampled_genes = rng.choice(
        test_gene_names,
        size=len(test_gene_names),
        replace=True
    )

    # Initialize lists for the observed and predicted response profiles
    # included in the current bootstrap sample
    true_profiles = []
    pred_profiles = []
    correlations = []

    # Collect the profiles belonging to each sampled perturbation
    for gene in sampled_genes:

        # Find all rows corresponding to the current perturbation.
        # Normally, these are its Control, IFNγ, and Co-culture profiles.
        idx = np.where(
            meta_test["perturbation"].values == gene
        )[0]

        # Add the observed and predicted log2FC profiles
        # for all conditions of the current perturbation
        true_profiles.append(Y_test[idx])
        pred_profiles.append(Y_pred[idx])

        # Calculate Pearson correlation separately for each
        # condition-specific response profile
        for i in idx:

            # Pearson correlation is undefined for constant profiles
            if (
                np.std(Y_test[i]) > 0
                and np.std(Y_pred[i]) > 0
            ):
                r, _ = pearsonr(
                    Y_test[i],
                    Y_pred[i]
                )
                correlations.append(r)

    # Combine all sampled perturbation-condition profiles
    # into matrices for the current bootstrap repetition
    true_profiles = np.vstack(true_profiles)
    pred_profiles = np.vstack(pred_profiles)

    # Calculate overall MAE for the current bootstrap sample
    bootstrap_mae.append(
        mean_absolute_error(
            true_profiles.ravel(),
            pred_profiles.ravel()
        )
    )

    # Calculate overall RMSE for the current bootstrap sample
    bootstrap_rmse.append(
        np.sqrt(
            mean_squared_error(
                true_profiles.ravel(),
                pred_profiles.ravel()
            )
        )
    )

    # Calculate the mean profile-level Pearson correlation
    # for the current bootstrap sample
    bootstrap_pearson.append(
        np.mean(correlations)
    )

In [77]:
# Define a function to calculate the 95% bootstrap confidence interval
# using the 2.5th and 97.5th percentiles of the bootstrap distribution
def bootstrap_ci(values):
    return np.percentile(
        values,
        [2.5, 97.5]
    )

# Calculate 95% confidence intervals for each performance metric
mae_ci = bootstrap_ci(bootstrap_mae)
rmse_ci = bootstrap_ci(bootstrap_rmse)
pearson_ci = bootstrap_ci(bootstrap_pearson)

# Display the original test-set performance together with
# the corresponding 95% bootstrap confidence intervals
print("95% bootstrap confidence intervals")

print(
    f"MAE: {mae:.3f} "
    f"[{mae_ci[0]:.3f}, {mae_ci[1]:.3f}]"
)

print(
    f"RMSE: {rmse:.3f} "
    f"[{rmse_ci[0]:.3f}, {rmse_ci[1]:.3f}]"
)

print(
    f"Pearson r: {np.nanmean(pearson_values):.3f} "
    f"[{pearson_ci[0]:.3f}, {pearson_ci[1]:.3f}]"
)

95% bootstrap confidence intervals
-----------------------------------
MAE: 1.939 [1.857, 2.004]
RMSE: 3.450 [3.352, 3.523]
Pearson r: 0.698 [0.678, 0.726]


In [80]:
# Other method Ridge regression model 
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV

# Use perturbation target as grouping variable
groups = meta_train["perturbation"].values

cv = GroupKFold(n_splits=5)

# Define Ridge hyperparameters
param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge = Ridge()

search_ridge = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

search_ridge.fit(
    X_train,
    Y_train_pca,
    groups=groups
)

print("Best Ridge parameters:")
print(search_ridge.best_params_)

Best Ridge parameters:
{'alpha': 10}


In [81]:
# Retrieve the Ridge regression model with the best regularization
# parameter identified during cross-validation
best_ridge = search_ridge.best_estimator_

# Use the optimized Ridge model to predict the 50 PCA components
# of the transcriptomic response for the held-out perturbations
Y_pred_pca_ridge = best_ridge.predict(X_test)

# Transform the predicted PCA components back into the original
# 2,000-gene log2FC space so that they can be directly compared
# with the observed test response profiles
Y_pred_ridge = pca_y.inverse_transform(
    Y_pred_pca_ridge
)

In [82]:
# Calculate the overall Mean Absolute Error (MAE) of the Ridge model
# across all held-out perturbation-condition profiles and response genes
mae_ridge = mean_absolute_error(
    Y_test.ravel(),
    Y_pred_ridge.ravel()
)

# Calculate the overall Root Mean Squared Error (RMSE).
# RMSE penalizes larger prediction errors more strongly than MAE.
rmse_ridge = np.sqrt(
    mean_squared_error(
        Y_test.ravel(),
        Y_pred_ridge.ravel()
    )
)

# Initialize a list to store the Pearson correlation
# for each held-out perturbation-condition profile
pearson_ridge = []

# Loop through all test profiles
for i in range(Y_test.shape[0]):

    # Check that both the observed and predicted profiles
    # contain variation, since Pearson correlation is undefined
    # for constant profiles
    if (
        np.std(Y_test[i]) > 0
        and np.std(Y_pred_ridge[i]) > 0
    ):

        # Calculate the Pearson correlation between the observed
        # and Ridge-predicted log2FC values across the 2,000 response genes
        r, _ = pearsonr(
            Y_test[i],
            Y_pred_ridge[i]
        )

    else:
        # Store NaN if Pearson correlation cannot be calculated
        r = np.nan

    # Store the correlation for the current test profile
    pearson_ridge.append(r)

# Display the overall Ridge prediction errors
print("Ridge MAE:", mae_ridge)
print("Ridge RMSE:", rmse_ridge)

# Display the mean Pearson correlation across all
# held-out perturbation-condition profiles
print(
    "Ridge mean Pearson r:",
    np.nanmean(pearson_ridge)
)

Ridge MAE: 2.020950443732414
Ridge RMSE: 3.5068199063641754
Ridge mean Pearson r: 0.6856135217751488


In [83]:
# Calculate uncertainty of the Ridge model using
# perturbation-level bootstrap resampling

rng = np.random.default_rng(42)

# Extract the 10 unique held-out perturbation genes
test_gene_names = meta_test["perturbation"].unique()

# Number of bootstrap repetitions
N_BOOTSTRAPS = 1000

bootstrap_mae_ridge = []
bootstrap_rmse_ridge = []
bootstrap_pearson_ridge = []

# Repeat bootstrap resampling 1,000 times
for b in range(N_BOOTSTRAPS):

    # Sample held-out perturbation genes with replacement.
    # All three condition-specific profiles of each perturbation
    # remain together in the bootstrap sample.
    sampled_genes = rng.choice(
        test_gene_names,
        size=len(test_gene_names),
        replace=True
    )

    true_profiles = []
    pred_profiles = []
    correlations = []

    for gene in sampled_genes:

        # Find the three condition-specific profiles
        # belonging to the current perturbation
        idx = np.where(
            meta_test["perturbation"].values == gene
        )[0]

        # Collect observed and Ridge-predicted profiles
        true_profiles.append(
            Y_test[idx]
        )

        pred_profiles.append(
            Y_pred_ridge[idx]
        )

        # Calculate Pearson correlation separately
        # for each condition-specific profile
        for i in idx:

            if (
                np.std(Y_test[i]) > 0
                and np.std(Y_pred_ridge[i]) > 0
            ):

                r, _ = pearsonr(
                    Y_test[i],
                    Y_pred_ridge[i]
                )

                correlations.append(r)

    # Combine all sampled profiles
    true_profiles = np.vstack(
        true_profiles
    )

    pred_profiles = np.vstack(
        pred_profiles
    )

    # Calculate MAE for the current bootstrap sample
    bootstrap_mae_ridge.append(
        mean_absolute_error(
            true_profiles.ravel(),
            pred_profiles.ravel()
        )
    )

    # Calculate RMSE for the current bootstrap sample
    bootstrap_rmse_ridge.append(
        np.sqrt(
            mean_squared_error(
                true_profiles.ravel(),
                pred_profiles.ravel()
            )
        )
    )

    # Calculate mean Pearson correlation
    # for the current bootstrap sample
    bootstrap_pearson_ridge.append(
        np.mean(correlations)
    )

In [84]:
# Calculate 95% bootstrap confidence intervals
mae_ci_ridge = bootstrap_ci(
    bootstrap_mae_ridge
)

rmse_ci_ridge = bootstrap_ci(
    bootstrap_rmse_ridge
)

pearson_ci_ridge = bootstrap_ci(
    bootstrap_pearson_ridge
)

# Display Ridge performance with uncertainty
print("Ridge regression")

print(
    f"MAE: {mae_ridge:.3f} "
    f"[{mae_ci_ridge[0]:.3f}, {mae_ci_ridge[1]:.3f}]"
)

print(
    f"RMSE: {rmse_ridge:.3f} "
    f"[{rmse_ci_ridge[0]:.3f}, {rmse_ci_ridge[1]:.3f}]"
)

print(
    f"Pearson r: {np.nanmean(pearson_ridge):.3f} "
    f"[{pearson_ci_ridge[0]:.3f}, {pearson_ci_ridge[1]:.3f}]"
)

Ridge regression
MAE: 2.021 [1.914, 2.113]
RMSE: 3.507 [3.387, 3.608]
Pearson r: 0.686 [0.662, 0.714]
